# MindCTRL — P300 Classifier: Train & Evaluate

Load a recorded `.xdf` session, train the XDawn+Riemannian classifier,
and inspect per-fold accuracy and the confusion matrix.

**Prerequisites**
```
cd ML_Pipeline
pip install -r requirements.txt
```

In [ ]:
import sys
sys.path.insert(0, '..')   # allow importing from ML_Pipeline root

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from bci_essentials.io.xdf_sources import XdfEegSource, XdfMarkerSource
from bci_essentials.bci_controller import BciController
from bci_essentials.paradigm.p300_paradigm import P300Paradigm
from bci_essentials.data_tank.data_tank import DataTank
from bci_essentials.classification.erp_rg_classifier import ErpRgClassifier

In [ ]:
# ── 1. Point to your XDF file ─────────────────────────────────────────────
XDF_PATH = "../data/session_001.xdf"   # <-- change this

In [ ]:
# ── 2. Load data & run offline training ───────────────────────────────────
eeg_source    = XdfEegSource(XDF_PATH)
marker_source = XdfMarkerSource(XDF_PATH)
paradigm      = P300Paradigm()
data_tank     = DataTank()

classifier = ErpRgClassifier()
classifier.set_p300_clf_settings(
    n_splits=5,
    lico_expansion_factor=4,
    remove_flats=True,
    random_seed=42,
)

controller = BciController(classifier, eeg_source, marker_source, paradigm, data_tank)
controller.setup(online=False)
controller.run()

print("Training complete.")

In [ ]:
# ── 3. Accuracy per fold ──────────────────────────────────────────────────
# ErpRgClassifier stores cross-val results in .offline_trial_results
if hasattr(classifier, 'offline_trial_results') and classifier.offline_trial_results:
    results = classifier.offline_trial_results
    accuracies = [r['accuracy'] for r in results if 'accuracy' in r]
    print(f"Mean accuracy: {np.mean(accuracies):.3f} ± {np.std(accuracies):.3f}")

    plt.figure(figsize=(7, 3))
    plt.bar(range(1, len(accuracies)+1), accuracies)
    plt.axhline(np.mean(accuracies), color='red', linestyle='--', label='mean')
    plt.ylim(0, 1)
    plt.xlabel('Fold')
    plt.ylabel('Accuracy')
    plt.title('P300 Classifier — Cross-validation Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No offline_trial_results found — check the classifier output above.")

In [ ]:
# ── 4. Confusion matrix (stimulus index) ─────────────────────────────────
# Index 0-15 = pitch buttons, 16 = Play/Pause
LABELS = (
    [f"Red-{n}"    for n in ['C','H','E','Y']] +
    [f"Blue-{n}"   for n in ['C','H','E','Y']] +
    [f"Yell-{n}"   for n in ['C','H','E','Y']] +
    [f"Grn-{n}"    for n in ['C','H','E','Y']] +
    ["Play/Pause"]
)

if hasattr(classifier, 'confusion_matrix') and classifier.confusion_matrix is not None:
    cm = classifier.confusion_matrix
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABELS[:cm.shape[1]],
                yticklabels=LABELS[:cm.shape[0]], ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title('Confusion Matrix')
    plt.tight_layout()
    plt.show()
else:
    print("No confusion matrix available.")

In [ ]:
# ── 5. Save model ─────────────────────────────────────────────────────────
SAVE_PATH = "../models/p300_classifier.joblib"
joblib.dump(classifier, SAVE_PATH)
print(f"Saved → {SAVE_PATH}")